# AWQ vs GPTQ vs FP8: the quantization trade-off, measured in vLLM

Companion notebook for GDE #7 (Nov 25, 2026). vLLM 0.33.0. It reproduces the
post's key arithmetic:

1. **Weight memory** per method (BF16 / FP8 / INT4 group-128) for 8B and 70B.
2. **Toy quantization error**: per-tensor vs per-group rounding, and the
   AWQ idea — scale the salient channels before rounding.
3. **Throughput / cost trade-off** from the Spheron Nov 24, 2026 study
   (Qwen3.8-27B, vLLM 0.34.0 as reported, RTX PRO 6000).
4. **vLLM 0.33 config guard**: which `--quantization` method runs on which
   GPU, and the Marlin W4A16 fallback trap.

All benchmark figures are the cited sources' own reported results, not
measurements from this machine. The arithmetic in cells 1–2 is computed here.


## 1. Weight memory: the formula is bytes per parameter

BF16 = 2 B/param. FP8 = 1 B/param (per-tensor scale is negligible).
INT4 group-128 = 0.5 B/param weights + one FP16 scale (2 B) per 128 weights
= 0.515625 B/param. Decimal GB, matching checkpoint file sizes.

In [1]:
def weight_gb(params_b, bytes_per_param):
    return params_b * bytes_per_param

METHODS = {"BF16": 2.0, "FP8 (W8A8)": 1.0, "INT4 g128 (AWQ/GPTQ)": 0.5 + 2 / 128}

for size in (8, 70):
    row = {m: round(weight_gb(size, b), 2) for m, b in METHODS.items()}
    print(f"{size}B model:", row)

# Asserts: the numbers printed in the post's first diagram.
assert weight_gb(8, METHODS["BF16"]) == 16.0
assert weight_gb(8, METHODS["FP8 (W8A8)"]) == 8.0
assert abs(weight_gb(8, METHODS["INT4 g128 (AWQ/GPTQ)"]) - 4.125) < 1e-9
assert weight_gb(70, METHODS["BF16"]) == 140.0
assert weight_gb(70, METHODS["FP8 (W8A8)"]) == 70.0
assert abs(weight_gb(70, METHODS["INT4 g128 (AWQ/GPTQ)"]) - 36.09375) < 1e-9
# The INT4 scale overhead is real but small: 3.1% over bare 4-bit.
assert abs(METHODS["INT4 g128 (AWQ/GPTQ)"] / 0.5 - 1.03125) < 1e-9
print("OK: 8B 16.0 / 8.0 / 4.1 GB; 70B 140.0 / 70.0 / 36.1 GB")
print("A 70B INT4 model fits on one 80 GB card; BF16 needs two.")

8B model: {'BF16': 16.0, 'FP8 (W8A8)': 8.0, 'INT4 g128 (AWQ/GPTQ)': 4.12}
70B model: {'BF16': 140.0, 'FP8 (W8A8)': 70.0, 'INT4 g128 (AWQ/GPTQ)': 36.09}
OK: 8B 16.0 / 8.0 / 4.1 GB; 70B 140.0 / 70.0 / 36.1 GB
A 70B INT4 model fits on one 80 GB card; BF16 needs two.


## 2. Toy quantization error: why groups beat tensors, and what AWQ adds

A toy 512×512 weight matrix. Rows have different scales (0.1x to 3x, the
thing that breaks naive quantization), and 1% of input channels carry 10x
activations — the AWQ observation that a few channels matter far more than
the rest. Round-to-nearest (RTN) at 4 bits, three ways:

- **per-tensor**: one scale for the whole matrix — the biggest row sets the
  grid, and small rows round to zero.
- **per-group g128**: one scale per (row, 128-column block) — the grid
  adapts to local scale.
- **AWQ-style**: before rounding, scale up the salient channels by `s`
  (grid-searched, as real AWQ does), quantize, then fold `1/s` back into
  the next layer's weights — mathematically identical output, finer grid
  where it matters.

Two metrics: plain relative MSE, and activation-weighted output error
(error × activation magnitude, squared) — the one that predicts what a
model actually outputs. This is a toy; real accuracy deltas are the evals
in section 3. But the ordering is the mechanism behind the AWQ paper's
Table 4: per-tensor worst, groups better, AWQ scaling better still.

In [2]:
import numpy as np

rng = np.random.default_rng(0)
# Rows have different scales (0.1x to 3x); 1% of channels carry 10x activations.
row_scale = np.exp(rng.uniform(np.log(0.1), np.log(3.0), (512, 1)))
W = rng.normal(0, 1, (512, 512)).astype(np.float64) * row_scale
salient = rng.random(512) < 0.01
act_scale = np.where(salient, 10.0, 1.0)          # per-channel activation magnitude

def rtn_int4(X, group):
    """Symmetric INT4 round-to-nearest. group=None: one scale for the whole
    matrix. group=128: one scale per (row, 128-column block), as in real
    AWQ/GPTQ group-128 checkpoints."""
    if group is None:
        s = np.abs(X).max() / 7.0                 # INT4 range [-7, 7]
        return np.round(X / s).clip(-7, 7) * s
    Xq = np.empty_like(X)
    for j in range(0, X.shape[1], group):
        block = X[:, j:j + group]
        s = np.abs(block).max(axis=1, keepdims=True) / 7.0
        Xq[:, j:j + group] = np.round(block / s).clip(-7, 7) * s
    return Xq

def rel_mse(A, B):
    return float(np.mean((A - B) ** 2) / np.mean(A ** 2))

def werr(A, B, act):
    # Activation-weighted relative output error: what the next layer sees.
    return float(np.sum(((A - B) * act) ** 2) / np.sum((A * act) ** 2))

act = act_scale[None, :]
err_tensor = rel_mse(W, rtn_int4(W, None))        # one scale for the matrix
err_group = rel_mse(W, rtn_int4(W, 128))          # one scale per (row, g128)
w_tensor = werr(W, rtn_int4(W, None), act)
w_group = werr(W, rtn_int4(W, 128), act)

# AWQ-style: grid-search one shared s for the salient channels (real AWQ
# searches s per channel to minimize error), then fold 1/s back.
best_s, best_e = None, None
for s in (1.0, 1.25, 1.5, 2.0, 2.5):
    svec = np.where(salient, s, 1.0)
    e = werr(W, rtn_int4(W * svec, 128) / svec, act)
    print(f"  AWQ-style s={s}: weighted err {e:.4f}")
    if best_e is None or e < best_e:
        best_s, best_e = s, e

print(f"plain rel. MSE  per-tensor: {err_tensor:.4f}  per-group g128: {err_group:.4f}  ({err_tensor / err_group:.1f}x better)")
print(f"weighted err   per-tensor: {w_tensor:.4f}  per-group g128: {w_group:.4f}")
print(f"weighted err   AWQ-style (best s={best_s}): {best_e:.4f}  ({w_group / best_e:.1f}x better than plain g128)")

assert err_group < err_tensor, "groups must beat per-tensor"
assert err_tensor / err_group > 5, "groups should be several x better here"
assert best_e < w_group, "AWQ scaling must beat plain groups on the weighted metric"
assert best_s == 2.0, "grid search should pick s=2.0 on this seed"
print("OK: per-tensor < per-group < AWQ-scaled, as in the AWQ paper's mechanism.")

  AWQ-style s=1.0: weighted err 0.0146
  AWQ-style s=1.25: weighted err 0.0118


  AWQ-style s=1.5: weighted err 0.0100
  AWQ-style s=2.0: weighted err 0.0093
  AWQ-style s=2.5: weighted err 0.0101
plain rel. MSE  per-tensor: 0.1008  per-group g128: 0.0138  (7.3x better)
weighted err   per-tensor: 0.1077  per-group g128: 0.0146
weighted err   AWQ-style (best s=2.0): 0.0093  (1.6x better than plain g128)
OK: per-tensor < per-group < AWQ-scaled, as in the AWQ paper's mechanism.


## 3. The measured trade-off: Spheron, Qwen3.8-27B, Nov 23–24, 2026

Source: Spheron Research, "FP8 vs NVFP4 vs AWQ: Which Qwen3.8-27B
Checkpoint to Serve?", measured Nov 23, 2026, published Nov 24, 2026.
Engine vLLM 0.34.0 as reported, one RTX PRO 6000 (Blackwell, 96 GB),
`vllm bench serve`, 1,024 in / 512 out, prefix caching off, median of 3
repeats. Latency target: 20 tok/s per user and first-token p95 < 2 s.
GPU billed $2.405/h. Accuracy: lm-evaluation-harness 0.4.13, temperature 0.

The cell checks the two claims the post makes: (a) no quantized checkpoint
lost a statistically significant amount of accuracy — every |delta| is
inside two paired standard errors; (b) the cost ranking at the latency
target is NVFP4 < FP8 < AWQ < BF16, and AWQ's cost is counted at 4 users
because its TTFT p95 (2.53 s) misses the target at 16.

In [3]:
# Accuracy: (BF16, FP8, NVFP4, AWQ) in percent; paired deltas with 2xSE.
evals = {
    "GSM8K (1,319)":      (96.1, 96.2, 96.2, 96.1, {"FP8": (0.1, 0.6), "NVFP4": (0.1, 0.7), "AWQ": (0.0, 0.8)}),
    "MMLU-Pro (1,960)":   (71.3, 72.0, 71.4, 71.4, {"FP8": (0.8, 1.3), "NVFP4": (0.1, 1.4), "AWQ": (0.1, 1.4)}),
    "MATH-500 (500)":     (78.4, 76.2, 76.4, 77.0, {"FP8": (-2.2, 2.4), "NVFP4": (-2.0, 2.8), "AWQ": (-1.4, 2.7)}),
    "IFEval (541)":       (81.7, 81.1, 83.5, 83.4, {"FP8": (-0.6, 2.0), "NVFP4": (1.8, 2.6), "AWQ": (1.7, 2.6)}),
}
print("Significance check (|delta| < 2x paired SE  =>  not significant):")
for name, (_, _, _, _, deltas) in evals.items():
    for method, (d, se) in deltas.items():
        sig = abs(d) > 2 * se
        print(f"  {name:18s} {method:6s} delta {d:+.1f} ± {se:.1f}  significant={sig}")
        assert not sig, f"{method} on {name} should not be significant"

# Mean accuracy change vs BF16 across the four evals (the decision chart's x-axis).
means = {}
for method in ("FP8", "NVFP4", "AWQ"):
    means[method] = sum(v[4][method][0] for v in evals.values()) / 4
print("\nMean accuracy change vs BF16:", {k: round(v, 2) for k, v in means.items()})
assert abs(means["FP8"] - (-0.475)) < 1e-9
assert abs(means["AWQ"] - 0.1) < 1e-9

# Throughput / cost at the latency target.
tput = {"BF16": (26.5, 1052, 6.62, 4), "FP8": (45.8, 1567, 1.21, 16),
        "NVFP4": (59.5, 1876, 0.95, 16), "AWQ": (69.4, 1292, 2.87, 4)}
print("\n1-user tok/s, peak tok/s (128 users), $/M at target, users at target:")
for m, (one, peak, cost, users) in tput.items():
    print(f"  {m:6s} {one:5.1f}  {peak:,}  ${cost:.2f}  @ {users} users")
assert tput["AWQ"][0] > tput["FP8"][0] > tput["BF16"][0]   # AWQ fastest single user
assert tput["NVFP4"][1] > tput["FP8"][1] > tput["AWQ"][1] > tput["BF16"][1]
assert tput["NVFP4"][2] < tput["FP8"][2] < tput["AWQ"][2] < tput["BF16"][2]
assert abs(tput["BF16"][2] / tput["FP8"][2] - 5.47) < 0.01  # FP8 5.5x cheaper than BF16
assert abs(tput["BF16"][2] / tput["NVFP4"][2] - 6.97) < 0.01
print("OK: AWQ fastest for one user; NVFP4 cheapest at load; FP8 5.5x cheaper than BF16.")

Significance check (|delta| < 2x paired SE  =>  not significant):
  GSM8K (1,319)      FP8    delta +0.1 ± 0.6  significant=False
  GSM8K (1,319)      NVFP4  delta +0.1 ± 0.7  significant=False
  GSM8K (1,319)      AWQ    delta +0.0 ± 0.8  significant=False
  MMLU-Pro (1,960)   FP8    delta +0.8 ± 1.3  significant=False
  MMLU-Pro (1,960)   NVFP4  delta +0.1 ± 1.4  significant=False
  MMLU-Pro (1,960)   AWQ    delta +0.1 ± 1.4  significant=False
  MATH-500 (500)     FP8    delta -2.2 ± 2.4  significant=False
  MATH-500 (500)     NVFP4  delta -2.0 ± 2.8  significant=False
  MATH-500 (500)     AWQ    delta -1.4 ± 2.7  significant=False
  IFEval (541)       FP8    delta -0.6 ± 2.0  significant=False
  IFEval (541)       NVFP4  delta +1.8 ± 2.6  significant=False
  IFEval (541)       AWQ    delta +1.7 ± 2.6  significant=False

Mean accuracy change vs BF16: {'FP8': -0.48, 'NVFP4': 0.0, 'AWQ': 0.1}

1-user tok/s, peak tok/s (128 users), $/M at target, users at target:
  BF16    26.5  1,052  

## 4. vLLM 0.33 config guard: pick the method, dodge the fallback

From the vLLM docs (quantization page, read Nov 25, 2026):

- Methods: `awq` / `awq_marlin` / `auto_awq`, `gptq` / `gptq_marlin` /
  `auto_gptq`, `fp8` (online, per-tensor static), `modelopt` (FP8),
  `compressed-tensors` (llm-compressor: FP8 W8A8, INT4 W4A16, INT8 W4A8/W8A8).
- **Marlin always runs 16-bit activations** in current vLLM — a 4-bit AWQ or
  GPTQ checkpoint on the Marlin path is weight-only (W4A16). The old
  `VLLM_MARLIN_INPUT_DTYPE` env var is removed (setting it is an error).
  For W4A8, use the `humming` method with `VLLM_HUMMING_INPUT_QUANT_CONFIG`.
- **Silent fallback**: if the kernel for a requested activation scheme is
  unavailable, vLLM falls back to a weight-only kernel (W8A8 runs as W8A16)
  and only logs a warning. `VLLM_STRICT_QUANT_SCHEME=1` forbids it.
- `fp_quant` and the QuTLASS backend are **removed** — use a
  compressed-tensors checkpoint instead.
- Hardware: AWQ Turing+, GPTQ Volta+, Marlin Turing+, llm-compressor FP8
  W8A8 Ada/Hopper (and AMD). No FP8 tensor cores on Ampere — FP8 KV/weights
  still save memory there, but activations dequantize to BF16.

The guard below encodes this and refuses the two traps: asking for
`fp_quant`, and asking for W4A8 without the humming method.

In [4]:
REMOVED = {"fp_quant": "removed in current vLLM — use a compressed-tensors checkpoint"}
MARLIN_ACTIVATION = "16-bit only: AWQ/GPTQ on Marlin is W4A16, weight-only"
SUPPORT = {  # (min compute capability) from the vLLM quantization support table
    "awq": ("Turing (7.5)", "Marlin"),
    "gptq": ("Volta (7.0)", "Marlin"),
    "fp8": ("Ada (8.9) for W8A8", "CUTLASS / DeepGEMM"),
    "modelopt": ("Ada (8.9)", "CUTLASS"),
}

def serve_command(model, method, gpu_cc, activation="W4A16", strict=False):
    if method in REMOVED:
        raise ValueError(f"{method} is {REMOVED[method]}")
    if activation == "W4A8" and method != "humming":
        raise ValueError("W4A8 needs the humming method "
                         "(VLLM_HUMMING_INPUT_QUANT_CONFIG); "
                         "Marlin cannot quantize activations.")
    if method in ("awq", "gptq") and activation == "W4A16":
        note = f"NOTE: {MARLIN_ACTIVATION} — weights shrink, activations stay BF16."
    else:
        note = ""
    cmd = (f"python -m vllm.entrypoints.openai.api_server --model {model} "
           f"--quantization {method} --dtype bfloat16")
    if strict:
        cmd = "VLLM_STRICT_QUANT_SCHEME=1 " + cmd
    return cmd, note

for m in ("awq", "gptq", "fp8"):
    print(f"{m:8s} min GPU {SUPPORT[m][0]:22s} kernel: {SUPPORT[m][1]}")

cmd, note = serve_command("Qwen/Qwen3-8B-AWQ", "awq", 9.0, strict=True)
print("\n" + cmd + "\n" + note)
cmd, _ = serve_command("Qwen/Qwen3-8B-FP8", "fp8", 9.0)
print(cmd + "   # online FP8: quantizes the BF16 checkpoint at load, no pre-quantized file needed")

try:
    serve_command("some-model", "fp_quant", 9.0)
except ValueError as e:
    print("\nGuard caught:", e)
try:
    serve_command("some-model", "awq", 9.0, activation="W4A8")
except ValueError as e:
    print("Guard caught:", e)
print("OK: config guard matches the vLLM 0.33 quantization docs.")

awq      min GPU Turing (7.5)           kernel: Marlin
gptq     min GPU Volta (7.0)            kernel: Marlin
fp8      min GPU Ada (8.9) for W8A8     kernel: CUTLASS / DeepGEMM

VLLM_STRICT_QUANT_SCHEME=1 python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-8B-AWQ --quantization awq --dtype bfloat16
NOTE: 16-bit only: AWQ/GPTQ on Marlin is W4A16, weight-only — weights shrink, activations stay BF16.
python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-8B-FP8 --quantization fp8 --dtype bfloat16   # online FP8: quantizes the BF16 checkpoint at load, no pre-quantized file needed

Guard caught: fp_quant is removed in current vLLM — use a compressed-tensors checkpoint
Guard caught: W4A8 needs the humming method (VLLM_HUMMING_INPUT_QUANT_CONFIG); Marlin cannot quantize activations.
OK: config guard matches the vLLM 0.33 quantization docs.


## Takeaways

- **Memory**: 8B weights are 16.0 GB (BF16) / 8.0 GB (FP8) / 4.1 GB (INT4
  g128); 70B is 140 / 70 / 36.1 GB. The INT4 scale overhead is 3.1%.
- **Accuracy**: on the same model, same prompts, same harness (Spheron,
  Qwen3.8-27B), FP8, NVFP4 and INT4-AWQ all sit within noise of BF16 —
  largest single-eval move is FP8's −2.2 pts on MATH-500, inside ±2.4 SE.
- **Throughput**: AWQ is fastest for one user (69.4 tok/s) but its TTFT
  under load misses a 2 s p95 target; FP8 is 5.5x cheaper per token than
  BF16 at the target; NVFP4 is cheapest at load on Blackwell.
- **AWQ vs GPTQ**: the AWQ paper's perplexity table has AWQ ahead by 0.09
  on Llama-2-7B INT4-g128 and tied on 70B — a small, model-dependent gap,
  not a rout. Pick by kernel support and your own evals, not by the name.
- **vLLM 0.33 traps**: Marlin is weight-only (W4A16); W4A8 needs humming;
  the W8A8→W8A16 fallback is silent unless `VLLM_STRICT_QUANT_SCHEME=1`;
  `fp_quant` is gone.

*All benchmark figures are the cited sources' own reported results
(Spheron Research Nov 24, 2026; AWQ paper arXiv:2306.00978 Table 4;
AIMultiple Qwen3-32B; Runcrate H100 FP8 test) — not independently measured
on this machine. The memory arithmetic, toy error demo, and config guard
above were computed/run here.*